In [2]:
import sys
from pathlib import Path

print("Python  :", sys.version.split()[0])
print("Folder  :", Path.cwd().name)

_missing = []
for _name in ['numpy', 'pandas', 'sklearn']:
    try:
        __import__(_name)
    except ImportError:
        _missing.append(_name)

for _name in ['numpy', 'pandas', 'sklearn']:
    _mark = "missing" if _name in _missing else "ok"
    print(f"  {_name:<14} {_mark}")

if _missing:
    print()
    print("STOP. Some libraries are missing:", ", ".join(_missing))
    print("Ask your instructor to run the setup in labs/SETUP.md.")
else:
    print()
    print("All good. You can carry on to Step 1.")

Python  : 3.13.15
Folder  : content
  numpy          ok
  pandas         ok
  sklearn        ok

All good. You can carry on to Step 1.


In [3]:
#dataset
import csv
from pathlib import Path
import numpy as np

SEED = 42
N_ROWS = 600

DATA = Path("delivery_times.csv")

def make_delivery_csv(path=DATA):
    rng = np.random.default_rng(SEED)

    distance_km = np.round(
        rng.uniform(0.5, 12.0, N_ROWS), 2
    )

    prep_time_min = np.round(
        rng.uniform(5, 30, N_ROWS), 0
    )

    traffic_level = rng.integers(1, 4, N_ROWS)

    rain = rng.binomial(1, 0.25, N_ROWS)

    delivery_min = np.round(
        6.0
        + 3.1 * distance_km
        + 0.65 * prep_time_min
        + 4.2 * traffic_level
        + 5.5 * rain
        + rng.normal(0, 2.5, N_ROWS),
        1
    )

    with path.open("w", newline="", encoding="utf-8") as fh:
        writer = csv.writer(fh)

        writer.writerow([
            "distance_km",
            "prep_time_min",
            "traffic_level",
            "rain",
            "delivery_min"
        ])

        for i in range(N_ROWS):
            writer.writerow([
                distance_km[i],
                int(prep_time_min[i]),
                int(traffic_level[i]),
                int(rain[i]),
                delivery_min[i]
            ])

    return path

make_delivery_csv()

print("Dataset created:", DATA)

Dataset created: delivery_times.csv


In [4]:
#load the dataset
import numpy as np
import pandas as pd

orders = pd.read_csv(DATA)

print("Rows, columns:", orders.shape)

print()

print(orders.head())

Rows, columns: (600, 5)

   distance_km  prep_time_min  traffic_level  rain  delivery_min
0         9.40             17              1     0          51.3
1         5.55             24              2     1          54.2
2        10.37             28              3     0          67.7
3         8.52             23              2     0          51.2
4         1.58             29              2     1          42.1


In [5]:
#Separate Features and Target
FEATURES = [
    "distance_km",
    "prep_time_min",
    "traffic_level",
    "rain"
]

TARGET = "delivery_min"

X = orders[FEATURES]
y = orders[TARGET]

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nFeatures:")
print(X.head(3))

print("\nTarget:")
print(y.head(3))

X shape: (600, 4)
y shape: (600,)

Features:
   distance_km  prep_time_min  traffic_level  rain
0         9.40             17              1     0
1         5.55             24              2     1
2        10.37             28              3     0

Target:
0    51.3
1    54.2
2    67.7
Name: delivery_min, dtype: float64


In [6]:
#train test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training:", len(X_train))
print("Testing :", len(X_test))
print("Total   :", len(X_train) + len(X_test))

Training: 480
Testing : 120
Total   : 600


In [7]:
#baseline
from sklearn.metrics import mean_absolute_error, mean_squared_error

average_time = y_train.mean()

print(
    f"Average delivery time: {average_time:.1f} minutes"
)

baseline_guesses = np.full(
    len(y_test),
    average_time
)

baseline_mae = mean_absolute_error(
    y_test,
    baseline_guesses
)

print(
    f"BASELINE MAE: {baseline_mae:.2f} minutes"
)

Average delivery time: 47.1 minutes
BASELINE MAE: 10.32 minutes


In [8]:
#train linear reg
from sklearn.linear_model import LinearRegression

model = LinearRegression()

model.fit(X_train, y_train)

print(
    "Model trained on",
    len(X_train),
    "orders."
)

Model trained on 480 orders.


In [9]:
#evaluate model
predictions = model.predict(X_test)

mae = mean_absolute_error(
    y_test,
    predictions
)

rmse = float(
    np.sqrt(
        mean_squared_error(
            y_test,
            predictions
        )
    )
)

print(f"BASELINE MAE : {baseline_mae:.2f} minutes")
print(f"MODEL MAE    : {mae:.2f} minutes")
print(f"MODEL RMSE   : {rmse:.2f} minutes")

BASELINE MAE : 10.32 minutes
MODEL MAE    : 1.92 minutes
MODEL RMSE   : 2.48 minutes


In [10]:
improvement = (
    100 * (baseline_mae - mae)
    / baseline_mae
)

print(
    f"The model is {improvement:.0f}% better than guessing."
)

The model is 81% better than guessing.


In [11]:
learned = pd.DataFrame({
    "feature": FEATURES,
    "minutes_added_per_unit": model.coef_.round(2)
})

print(learned.to_string(index=False))

print(
    f"\nIntercept: {model.intercept_:.1f} minutes"
)

      feature  minutes_added_per_unit
  distance_km                    3.07
prep_time_min                    0.65
traffic_level                    4.13
         rain                    5.55

Intercept: 6.4 minutes


In [12]:
#predict the order
new_order = pd.DataFrame([{
    "distance_km": 5.0,
    "prep_time_min": 20,
    "traffic_level": 2,
    "rain": 0
}])

minutes = model.predict(new_order)[0]

print(
    f"Predicted delivery time: {minutes:.1f} minutes"
)

Predicted delivery time: 43.0 minutes


In [13]:
# Predict the median for every test order
median_time = y_train.median()

median_guesses = np.full(
    len(y_test),
    median_time
)

T1_median_mae = mean_absolute_error(
    y_test,
    median_guesses
)

# Decide which baseline is better
if T1_median_mae < baseline_mae:
    T1_which_is_better = "median"
else:
    T1_which_is_better = "mean"

print("Median baseline MAE:", T1_median_mae)
print("Better baseline:", T1_which_is_better)

Median baseline MAE: 10.331666666666669
Better baseline: mean


In [14]:
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=7
)

In [15]:
model2 = LinearRegression()

model2.fit(
    X_tr2,
    y_tr2
)

LinearRegression()

In [16]:
model2 = LinearRegression().fit(
    X_tr2,
    y_tr2
)

In [17]:
predictions2 = model2.predict(X_te2)

In [20]:
T2_mae = mean_absolute_error(
    y_te2,
    predictions2
)

print(
    "70/30 split, seed 7 -> MAE:",
    T2_mae
)
len(X_te2)

70/30 split, seed 7 -> MAE: 2.096334872408644


180

In [21]:
def predict_minutes(
    distance_km,
    prep_time_min,
    traffic_level,
    rain
):

    new_order = pd.DataFrame([{
        "distance_km": distance_km,
        "prep_time_min": prep_time_min,
        "traffic_level": traffic_level,
        "rain": rain
    }])

    prediction = model.predict(new_order)[0]

    return round(prediction, 1)

In [22]:
T3_rainy_order = predict_minutes(
    3.0,
    15,
    1,
    1
)

print(
    "Rainy 3 km order ->",
    T3_rainy_order,
    "minutes"
)

Rainy 3 km order -> 35.0 minutes


In [23]:
# T1: Median baseline

median_time = y_train.median()

median_guesses = np.full(
    len(y_test),
    median_time
)

T1_median_mae = mean_absolute_error(
    y_test,
    median_guesses
)

if T1_median_mae < baseline_mae:
    T1_which_is_better = "median"
else:
    T1_which_is_better = "mean"

print("median baseline MAE:", T1_median_mae)
print("better baseline:", T1_which_is_better)

median baseline MAE: 10.331666666666669
better baseline: mean


In [24]:
# T2: 70/30 split with random_state=7

X_tr2, X_te2, y_tr2, y_te2 = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=7
)

model2 = LinearRegression()

model2.fit(
    X_tr2,
    y_tr2
)

predictions2 = model2.predict(X_te2)

T2_mae = mean_absolute_error(
    y_te2,
    predictions2
)

print("70/30 split, seed 7 -> MAE", T2_mae)

70/30 split, seed 7 -> MAE 2.096334872408644


In [25]:
# T3: Prediction function

def predict_minutes(
    distance_km,
    prep_time_min,
    traffic_level,
    rain
):

    new_order = pd.DataFrame([{
        "distance_km": distance_km,
        "prep_time_min": prep_time_min,
        "traffic_level": traffic_level,
        "rain": rain
    }])

    prediction = model.predict(new_order)[0]

    return round(prediction, 1)


T3_rainy_order = predict_minutes(
    3.0,
    15,
    1,
    1
)

print(
    "rainy 3 km order ->",
    T3_rainy_order,
    "minutes"
)

rainy 3 km order -> 35.0 minutes
